# The Model and the Likelihood

Model A, worked end to end on a single configuration

Dan Yavorsky  
Geoffery Zheng  
September 14, 2026

## What this notebook does

This notebook is the walkthrough. It takes one parameter configuration of Model A and follows it the whole way: draw a design, simulate the behavioral process, write out the joint likelihood, maximize it, and check what comes back.

Everything is self-contained, and the code is deliberately the simplest that will work. It handles Model A only and one configuration only, so nothing is generalized before it has been shown once. Notebook 02 takes the same machinery, generalizes it to both links, and runs it across configurations and repeated samples.

The functions are copies of `R/lib/dgp.R` and `R/lib/likelihood.R`, unrolled and commented so the mechanics are visible without opening the archive.

A word on why the data are simulated behaviorally rather than from the closed form. If we drew responses from `alpha_w^S - alpha_{w-1}^S` and then fit `alpha_w^S - alpha_{w-1}^S`, recovery would confirm only that `optim` works. By drawing utilities, taking an argmax, and applying the reporting rule, recovery tests the derivation itself: the claim that the maximum of Gumbel utilities is Gumbel with location equal to the log-sum inclusive value, and that it is independent of which alternative attained it.

This notebook writes no results files. Nothing in the article depends on it, so it can be edited freely.

## 1. The model in one screen

A consumer faces $J$ profiles. Profile $j$ carries attributes $x_j$ and utility

$$
u_j = x_j'\beta + \varepsilon_j, \qquad \varepsilon_j \sim \text{Gumbel}(0, 1) \text{ iid}.
$$

**First response.** She is asked which she prefers and picks the best one, $j^* = \arg\max_j u_j$. This is the usual multinomial logit.

**Second response.** She is asked how likely she is to buy it. She knows everything about the profiles on screen, but not her valuation of the outside good $\eta_0 \sim \text{Gumbel}(0,1)$. So what she holds is not a decision but a probability,

$$
p = \Pr(\eta_0 < u^* \mid u^*) = F(u^*), \qquad u^* = \max_j u_j,
$$

where $F(z) = \exp(-e^{-z})$ is the standard Gumbel CDF.

**Model A** says she reports the interval containing that probability. Given cut points $0 = \alpha_0 < \alpha_1 < \cdots < \alpha_{W-1} < \alpha_W = 1$ on the probability scale,

$$
y = w \iff p \in [\alpha_{w-1}, \alpha_w).
$$

From the researcher’s seat $u^*$ is random. The key lemma is that $u^* \sim \text{Gumbel}(\overline{\mu}, 1)$ with $\overline{\mu} = \ln S$ and $S = \sum_j e^{V_j}$, and that $u^*$ is independent of $j^*$. Evaluating that CDF at $F^{-1}(\alpha)$ collapses to a power, which gives the whole second-stage likelihood:

$$
\Pr(y = w) = \alpha_w^{\,S} - \alpha_{w-1}^{\,S}.
$$

Equivalently, from the researcher’s perspective the consumer’s purchase probability is $\text{Beta}(S, 1)$ distributed, with the inclusive value as its only shape parameter.

## 2. A design

We simulate two three-level attributes, dummy coded against a reference level, plus a continuous (e.g., price) attribute.

In [ ]:
#| label: design

make_design <- function(n_tasks, J, seed, intercept = FALSE) {
  set.seed(seed)
  n_rows <- n_tasks * J
  a     <- sample(1:3, n_rows, replace = TRUE)
  b     <- sample(1:3, n_rows, replace = TRUE)
  price <- runif(n_rows, 0.5, 2.5)
  X <- cbind(
    a2    = as.numeric(a == 2),
    a3    = as.numeric(a == 3),
    b2    = as.numeric(b == 2),
    b3    = as.numeric(b == 3),
    price = price
  )
  if (intercept) X <- cbind(X, const = 1)
  list(X = X, n_tasks = n_tasks, J = J, P = ncol(X))
}

N_TASKS <- 20*1000 # 20,000 tasks
J       <- 4       # 4 alternatives per task
W       <- 5       # a five-point purchase-likelihood scale

design <- make_design(N_TASKS, J, seed = 101)
head(design$X, 8)

     a2 a3 b2 b3     price
[1,]  0  0  0  1 0.6593832
[2,]  0  0  0  1 2.1495860
[3,]  1  0  0  1 1.7999991
[4,]  0  1  1  0 2.4604207
[5,]  0  1  0  1 1.1379471
[6,]  0  0  0  1 2.1380531
[7,]  1  0  0  1 1.0162167
[8,]  0  1  0  0 0.5251169

One task is a set of rows: rows 1 to 4 are the four profiles of task 1, rows 5 to 8 are task 2, and so on.

## 3. True parameters

We set the scale labels on the probability scale to make Model A interpretable: $\alpha_1 = 0.10$ says that answering “1” means a purchase probability below ten percent.

In [ ]:
#| label: truth

beta_true  <- c(a2 = 0.8, a3 = -0.5, b2 = 0.4, b3 = 1.0, price = -0.9)
alpha_true <- c(0.10, 0.30, 0.60, 0.85)

# Cut points live on the utility scale internally, so we move between the two
# scales with the standard Gumbel CDF and its inverse. These are exact
# inverses of each other and are used in both directions throughout.
alpha_to_cut <- function(a) -log(-log(a))   # probability -> utility,  F^{-1}
cut_to_alpha <- function(c) exp(-exp(-c))   # utility -> probability,  F
cut_true <- alpha_to_cut(alpha_true)

rbind(alpha = alpha_true, cut = cut_true)

            [,1]       [,2]     [,3]     [,4]
alpha  0.1000000  0.3000000 0.600000 0.850000
cut   -0.8340324 -0.1856268 0.671727 1.816961

## 4. Simulate the behavioral process

We draw utilities, take the argmax, convert the winning utility into the consumer’s purchase probability, and bin it.

In [ ]:
#| label: simulate

rgumbel <- function(n) -log(-log(runif(n)))

simulate_modelA <- function(design, beta, cut, seed) {
  set.seed(seed)
  n <- design$n_tasks
  J <- design$J

  # Deterministic utilities, reshaped to one row per task.
  # Requires J to be constant across consumers
  V <- matrix(design$X %*% beta, nrow = n, ncol = J, byrow = TRUE)

  # Add iid Gumbel noise
  u <- V + matrix(rgumbel(n * J), n, J)

  # Identify alternative with max utility per task
  jstar <- max.col(u, ties.method = "first")

  # The utility from jstar
  # (code operates as a matrix subset with a two-col matrix of positions)
  ustar <- u[cbind(seq_len(n), jstar)]

  # Model A: she reports the interval holding p = F(ustar). Binning p against
  # alpha is the same as binning ustar against cut, since F is increasing.
  y <- findInterval(ustar, cut) + 1L

  list(design = design, jstar = jstar, y = y, W = length(cut) + 1L,
       ustar = ustar, V = V)
}

dat <- simulate_modelA(design, beta_true, cut_true, seed = 501)

### What the two responses look like

The first response is roughly uniform because the design randomizes attributes across positions.

In [ ]:
#| label: responses_jstar

round(tabulate(dat$jstar, J) / N_TASKS, 4)   # chosen profile

[1] 0.2518 0.2520 0.2516 0.2445

The second response is not, and its shape is what the cut points control.

In [ ]:
#| label: responses_y

round(tabulate(dat$y,     W) / N_TASKS, 4)   # scale point

[1] 0.0190 0.0707 0.2170 0.3572 0.3361

### The lemma is visible in the simulated data

The two claims from Lemma 1 underlie the whole factorization. Both can be read straight off the draws, before any estimation.

In [ ]:
#| label: lemma-check

S     <- rowSums(exp(dat$V))
mubar <- log(S)

# (i) ustar is Gumbel(mubar, 1). Standardize and compare to a standard Gumbel.
z <- dat$ustar - mubar
cat("mean of standardized ustar:", round(mean(z), 4),
    " (Euler-Mascheroni = 0.5772)\n")

mean of standardized ustar: 0.5856  (Euler-Mascheroni = 0.5772)

sd   of standardized ustar: 1.2827  (pi/sqrt(6) = 1.2825)

KS test vs standard Gumbel, p = 0.852 

     1      2      3      4 
0.5912 0.5669 0.5773 0.6078 


    Kruskal-Wallis rank sum test

data:  z and factor(dat$jstar)
Kruskal-Wallis chi-squared = 4.2876, df = 3, p-value = 0.232

The group means sit within sampling error of one another and the test does not reject. That independence is what lets the joint likelihood factor into a choice term and an ordinal term: it says a consumer who picked profile 3 is no more or less enthusiastic, on average, than one who picked profile 1.

## 5. The likelihood

The joint probability of one task is the product of the two pieces:

$$
\underbrace{\frac{e^{V_{j^*}}}{S}}_{\text{first response}}
\times
\underbrace{\left( \alpha_{y}^{\,S} - \alpha_{y-1}^{\,S} \right)}_{\text{second response}}.
$$

The cut points must satisfy $c_1 < c_2 < \cdots < c_{W-1}$, which is an awkward constraint to hand an optimizer. We remove it by reparameterizing. Let

$$
\theta_1 = c_1, \qquad
\theta_m = \ln\left( c_m - c_{m-1} \right), \quad m = 2, \ldots, W-1,
$$

so the working parameter vector is $\left( \bfbeta, \theta_1, \ldots, \theta_{W-1} \right) \in \mathbb{R}^{P + W - 1}$, entirely unconstrained. Inverting gives the map `par_to_cut()` implements:

$$
c_1 = \theta_1, \qquad
c_w = \theta_1 + \sum_{m=2}^{w} e^{\theta_m}, \quad w = 2, \ldots, W-1 .
$$

Because $e^{\theta_m} > 0$ for every real $\theta_m$, the gaps are positive whatever the optimizer proposes, so monotonicity holds by construction.

One consequence to carry forward: standard errors come out on the $\theta$ scale, so reporting them for $\mathbf{c}$ needs the delta method. The Jacobian is read straight off the sum above,

$$
\frac{\partial c_w}{\partial \theta_1} = 1,
\qquad
\frac{\partial c_w}{\partial \theta_m} =
\begin{cases}
e^{\theta_m} & 2 \le m \le w, \\
0 & m > w,
\end{cases}
$$

which is the lower-triangular matrix `Jc` built in `fit_dual_mle()` below.

In [ ]:
#| label: likelihood

par_to_cut <- function(par, P, W) {
  if (W == 2) return(par[P + 1])
  par[P + 1] + c(0, cumsum(exp(par[(P + 2):(P + W - 1)])))
}
cut_to_par <- function(cut) c(cut[1], log(diff(cut)))

negloglik <- function(par, dat) {
  design <- dat$design
  P <- design$P
  W <- dat$W
  n <- design$n_tasks
  J <- design$J

  beta <- par[1:P]
  cut  <- par_to_cut(par, P, W)

  V     <- matrix(design$X %*% beta, nrow = n, ncol = J, byrow = TRUE)
  m     <- do.call(pmax, as.data.frame(V))     # row maxima
  logS  <- m + log(rowSums(exp(V - m)))        # log inclusive value

  # First response: multinomial logit log-probability of the chosen profile.
  ll_choice <- V[cbind(seq_len(n), dat$jstar)] - logS

  # Second response: difference of Gumbel CDFs at the standardized cut points.
  caug <- c(-Inf, cut, Inf)
  lo <- caug[dat$y]      - logS
  hi <- caug[dat$y + 1L] - logS
  p_ord <- exp(-exp(-hi)) - exp(-exp(-lo))

  -(sum(ll_choice) + sum(log(pmax(p_ord, 1e-312))))
}

A note on that second-response line (defining `p_ord`). Writing it as a plain difference of two `exp(-exp(-z))` terms is clear but loses precision when both cut points sit far into a tail and the two CDFs nearly cancel. The production code uses the algebraically equivalent but numerically stable rearrangement $e^{-e^{-h}}\left(1 - e^{-(e^{-l} - e^{-h})}\right)$.

## 6. Estimate

Starting values: $\beta = 0$, and cut points backed out of the observed category frequencies through the link.

In [ ]:
#| label: fit

fit_modelA <- function(dat) {
  P <- dat$design$P
  W <- dat$W

  freq <- tabulate(dat$y, nbins = W)
  cumq <- cumsum(freq)[1:(W - 1)] / sum(freq)
  cut0 <- log(dat$design$J) + alpha_to_cut(cumq)
  start <- c(rep(0, P), cut_to_par(cut0))

  fn  <- function(p) negloglik(p, dat)
  opt <- optim(start, fn, method = "BFGS",
               control = list(maxit = 1000, reltol = 1e-12))
  H   <- optimHess(opt$par, fn)      # observed information
  Vc  <- solve(H)

  cut_hat <- par_to_cut(opt$par, P, W)

  # Delta method for the cut points, which are a nonlinear function of the
  # working parameters.
  Jc <- matrix(0, W - 1, length(opt$par))
  Jc[, P + 1] <- 1
  if (W > 2) {
    d <- exp(opt$par[(P + 2):(P + W - 1)])
    for (w in 2:(W - 1)) Jc[w, (P + 2):(P + w)] <- d[1:(w - 1)]
  }

  list(beta = opt$par[1:P], se_beta = sqrt(diag(Vc)[1:P]),
       cut = cut_hat, se_cut = sqrt(diag(Jc %*% Vc %*% t(Jc))),
       nll = opt$value, convergence = opt$convergence,
       eigen = eigen(H, symmetric = TRUE, only.values = TRUE)$values,
       par = opt$par)
}

fit <- fit_modelA(dat)
cat("converged:", fit$convergence == 0, "\n")

converged: TRUE 

## 7. Confirm parameter recovery

In [ ]:
#| label: recovery

recovery <- data.frame(
  parameter = c(names(beta_true), paste0("c", 1:(W - 1))),
  truth     = c(beta_true, cut_true),
  estimate  = c(fit$beta, fit$cut),
  std_error = c(fit$se_beta, fit$se_cut)
)
recovery$z <- (recovery$estimate - recovery$truth) / recovery$std_error
knitr::kable(recovery, digits = 4)

  parameter       truth   estimate   std_error         z
  ----------- --------- ---------- ----------- ---------
  a2             0.8000     0.7989      0.0169   -0.0630
  a3            -0.5000    -0.5122      0.0205   -0.5969
  b2             0.4000     0.4088      0.0197    0.4467
  b3             1.0000     1.0184      0.0186    0.9929
  price         -0.9000    -0.9115      0.0133   -0.8659
  c1            -0.8340    -0.8407      0.0289   -0.2319
  c2            -0.1856    -0.1951      0.0261   -0.3628
  c3             0.6717     0.6553      0.0255   -0.6470
  c4             1.8170     1.8036      0.0267   -0.5028


Every $z$ statistic is small. With nine parameters we would expect roughly one value above two in absolute terms about forty percent of the time.

### The cut points on the probability scale

Pushing the estimated cut points back through the link turns scale labels into purchase probabilities, with standard errors.

In [ ]:
#| label: alpha

alpha_hat <- cut_to_alpha(fit$cut)
# delta method: d(alpha)/dc = alpha * exp(-c), and exp(-c) = -log(alpha)
se_alpha  <- fit$se_cut * alpha_hat * (-log(alpha_hat))

knitr::kable(
  data.frame(
    label      = paste("answer", 1:(W - 1), "or below"),
    alpha_true = alpha_true,
    alpha_hat  = alpha_hat,
    std_error  = se_alpha
  ), digits = 4)

  label                 alpha_true   alpha_hat   std_error
  ------------------- ------------ ----------- -----------
  answer 1 or below           0.10      0.0985      0.0066
  answer 2 or below           0.30      0.2966      0.0094
  answer 3 or below           0.60      0.5949      0.0079
  answer 4 or below           0.85      0.8481      0.0037


The cut points are fixed across tasks, so a label means the same thing on every screen: a respondent choosing the top box is reporting a purchase probability above 0.848, against a true 0.85.

## 8. Diagnostics

**Is the truth inside the confidence region?** Compare twice the log-likelihood gap between the truth and the MLE against a $\chi^2_9$ critical value.

In [ ]:
#| label: lrtest

nll_truth <- negloglik(c(beta_true, cut_to_par(cut_true)), dat)
lr   <- 2 * (nll_truth - fit$nll)
npar <- length(fit$par)
cat(sprintf("LR statistic %.2f vs chi-square(%d) 95%% critical value %.2f -> %s\n",
            lr, npar, qchisq(0.95, npar),
            ifelse(lr < qchisq(0.95, npar), "inside the region", "OUTSIDE")))

LR statistic 3.44 vs chi-square(9) 95% critical value 16.92 -> inside the region

**Is the information matrix well conditioned?** Identification is established analytically in the paper; what this checks is that the implementation inherits it. All eigenvalues positive means the negative-likelihood curves upward in every direction, and the smallest one bounds the precision of the worst-determined parameter combination.

In [ ]:
#| label: eigen

cat("smallest eigenvalue:", format(min(fit$eigen), digits = 4), "\n")

smallest eigenvalue: 816.5 

condition number:    90.02 

## Where this goes next

| notebook | what it adds |
|------------------------------------|------------------------------------|
| 02 | the same model generalized to both links, run across three configurations and forty independent datasets |
| 03 | the hierarchical sampler, its calibration, and a cross-check against an independent implementation |
| Appendix E | the same aggregate model estimated in Apollo, as a third independent check |